# 🎮 Steam Reviews — Kapsamlı Keşifsel Veri Analizi (EDA)

**Veri Kaynağı:** Delta Lake Silver Katmanı (`/delta/silver/steam_reviews`)  
**Kolonlar:** `app_id`, `app_name`, `review_text`, `review_score` (label: 0/1), `review_votes`, `timestamp`

---

## İçindekiler
1. Kütüphaneler ve Veri Okuma  
2. Temel İstatistikler  
3. Eksik Değer Analizi  
4. Sınıf Dağılımı (Class Distribution)  
5. Zaman Serisi Analizi  
6. Metin Analizi  
7. Korelasyon Analizi

In [ ]:
# ── 1. Kütüphaneler ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from collections import Counter
import re

# Matplotlib Türkçe karakter ve stil ayarları
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.style.use("seaborn-v0_8-whitegrid")

print("Kütüphaneler yüklendi ✓")

In [ ]:
# ── Spark oturumu ve Delta Lake'ten veri okuma ────────────────────────────────
spark = SparkSession.builder \
    .appName("SteamReviews_EDA") \
    .master("local[*]") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Silver katmandan oku
DELTA_PATH = "/delta/silver/steam_reviews"
df_spark = spark.read.format("delta").load(DELTA_PATH)

print(f"Schema:")
df_spark.printSchema()
print(f"\nToplam satır sayısı (Spark): {df_spark.count():,}")

In [ ]:
# ── Pandas'a dönüştürme ───────────────────────────────────────────────────────
# Analiz kolaylığı için Pandas DataFrame'e çeviriyoruz
df = df_spark.toPandas()

# timestamp kolonunu datetime'a çevir
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# review_score'u integer yap (0/1)
df["review_score"] = df["review_score"].astype("Int64")

print(f"Pandas DataFrame boyutu: {df.shape}")
df.head()

---
## 2. Temel İstatistikler
Veri setinin genel yapısını ve boyutunu anlamamız için ilk adım.

In [ ]:
# ── 2a. Genel bakış metrikleri ────────────────────────────────────────────────
toplam_kayit = len(df)
benzersiz_oyun = df["app_name"].nunique()
tarih_min = df["timestamp"].min()
tarih_max = df["timestamp"].max()

pozitif_sayi = (df["review_score"] == 1).sum()
negatif_sayi = (df["review_score"] == 0).sum()
pozitif_oran = pozitif_sayi / toplam_kayit * 100
negatif_oran = negatif_sayi / toplam_kayit * 100

print("=" * 55)
print("         TEMEL İSTATİSTİKLER")
print("=" * 55)
print(f"  Toplam kayıt sayısı      : {toplam_kayit:>12,}")
print(f"  Benzersiz oyun sayısı    : {benzersiz_oyun:>12,}")
print(f"  Tarih aralığı            : {tarih_min}")
print(f"                          → {tarih_max}")
print(f"  Pozitif yorum oranı      : {pozitif_oran:>11.2f}%  ({pozitif_sayi:,})")
print(f"  Negatif yorum oranı      : {negatif_oran:>11.2f}%  ({negatif_sayi:,})")
print("=" * 55)

In [ ]:
# ── 2b. df.describe() — güzel formatlı tablo ─────────────────────────────────
# Sayısal kolonların özet istatistikleri
desc = df.describe(include="all").T

# Sadece anlamlı sayısal kolonları filtrele
num_cols = ["app_id", "review_score", "review_votes"]
desc_num = df[num_cols].describe().T
desc_num = desc_num.round(2)

# Pandas Styler ile güzel görüntüleme
desc_num.style \
    .set_caption("Sayısal Kolonların Özet İstatistikleri") \
    .format("{:,.2f}") \
    .set_properties(**{"text-align": "right"}) \
    .background_gradient(cmap="Blues", subset=["mean", "std"])

**Yorum:** Yukarıdaki tablo, sayısal değişkenlerin ortalama, standart sapma ve çeyreklik değerlerini göstermektedir. `review_votes` kolonundaki yüksek standart sapma, oy dağılımının oldukça çarpık olduğuna işaret eder.

---
## 3. Eksik Değer Analizi
Modelleme öncesi hangi kolonlarda ne kadar eksik veri olduğunu belirlememiz kritik.

In [ ]:
# ── 3a. Eksik değer tablosu ───────────────────────────────────────────────────
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({
    "Kolon": df.columns,
    "Eksik Sayısı": null_counts.values,
    "Eksik Oranı (%)": null_pct.values
}).sort_values("Eksik Oranı (%)", ascending=False).reset_index(drop=True)

print("Eksik Değer Tablosu")
print("-" * 45)
null_df.style \
    .set_caption("Her Kolon İçin Null Oranı") \
    .bar(subset=["Eksik Oranı (%)"], color="#ff6b6b") \
    .format({"Eksik Oranı (%)": "{:.2f}%"})

In [ ]:
# ── 3b. Eksik değer heatmap'i ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Eksik değer matrisini oluştur (True=eksik → 1, False=mevcut → 0)
missing_matrix = df.isnull().astype(int)

# Eğer veri çok büyükse rastgele örnekle al
sample_size = min(len(df), 2000)
missing_sample = missing_matrix.sample(n=sample_size, random_state=42).reset_index(drop=True)

im = ax.imshow(missing_sample.T, aspect="auto", cmap="YlOrRd", interpolation="nearest")

ax.set_yticks(range(len(df.columns)))
ax.set_yticklabels(df.columns, fontsize=11)
ax.set_xlabel("Satır İndeksi (örneklem)")
ax.set_title("Eksik Değer Heatmap'i (Sarı = Mevcut, Kırmızı = Eksik)", fontsize=14)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["Mevcut", "Eksik"])

plt.tight_layout()
plt.show()

**Yorum:** Heatmap üzerinde kırmızı bölgeler eksik değerleri temsil eder. Eğer belirli kolonlarda yoğun kırmızı bant görülüyorsa, o kolonun modelleme öncesi doldurulması veya çıkarılması gerekebilir.

---
## 4. Sınıf Dağılımı (Class Distribution)
Sentiment sınıflandırması için sınıf dengesini anlamak büyük önem taşır.

In [ ]:
# ── 4a. Pie chart — Pozitif vs Negatif oran ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Pie Chart ---
labels = ["Pozitif (1)", "Negatif (0)"]
sizes = [pozitif_sayi, negatif_sayi]
colors = ["#2ecc71", "#e74c3c"]
explode = (0.03, 0.03)

axes[0].pie(
    sizes, explode=explode, labels=labels, colors=colors,
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 12},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
axes[0].set_title("Pozitif vs Negatif Yorum Oranı", fontsize=14)

# --- Bar Chart: Sınıf dağılımı ---
bars = axes[1].bar(labels, sizes, color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Sınıf Dağılımı (Adet)", fontsize=14)
axes[1].set_ylabel("Yorum Sayısı")
# Bar üstüne değer yaz
for bar, val in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.005,
                 f"{val:,}", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

# Sınıf dengesizliği kontrolü
ratio = max(pozitif_sayi, negatif_sayi) / min(pozitif_sayi, negatif_sayi) if min(pozitif_sayi, negatif_sayi) > 0 else float("inf")
if ratio > 2:
    print(f"⚠️  SINIF DENGESİZLİĞİ TESPİT EDİLDİ! Oran: {ratio:.2f}:1")
    print("   → Modelleme aşamasında oversampling, undersampling veya class_weight kullanılması önerilir.")
else:
    print(f"✓ Sınıf dağılımı nispeten dengeli. Oran: {ratio:.2f}:1")

In [ ]:
# ── 4b. Bar chart — En çok yorumlanan top-20 oyun ────────────────────────────
top20_games = df["app_name"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    top20_games.index[::-1],
    top20_games.values[::-1],
    color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)),
    edgecolor="white", linewidth=0.5
)

# Bar yanına değer yaz
for bar, val in zip(bars, top20_games.values[::-1]):
    ax.text(bar.get_width() + max(top20_games.values)*0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)

ax.set_xlabel("Yorum Sayısı")
ax.set_title("En Çok Yorumlanan Top-20 Oyun", fontsize=14)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

**Yorum:** Popüler oyunlar veri setinde baskın olabilir. Eğer tek bir oyun toplam verinin büyük kısmını oluşturuyorsa, model bu oyuna aşırı uyum sağlayabilir (overfitting). Stratified sampling kullanmak faydalı olabilir.

---
## 5. Zaman Serisi Analizi
Yorumların zaman içindeki dağılımını inceliyoruz.

In [ ]:
# ── 5a. Günlük yorum sayısı line chart ────────────────────────────────────────
# timestamp'ten tarih ve saat bilgilerini türet
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["year_week"] = df["timestamp"].dt.to_period("W")

daily_counts = df.groupby("date").size().reset_index(name="yorum_sayisi")
daily_counts["date"] = pd.to_datetime(daily_counts["date"])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_counts["date"], daily_counts["yorum_sayisi"],
        color="#3498db", linewidth=0.8, alpha=0.7)

# 7 günlük hareketli ortalama
if len(daily_counts) >= 7:
    daily_counts["ma_7"] = daily_counts["yorum_sayisi"].rolling(7).mean()
    ax.plot(daily_counts["date"], daily_counts["ma_7"],
            color="#e74c3c", linewidth=2, label="7 Günlük Hareketli Ort.")
    ax.legend(fontsize=11)

ax.set_xlabel("Tarih")
ax.set_ylabel("Yorum Sayısı")
ax.set_title("Günlük Yorum Sayısı Trendi", fontsize=14)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Saatlik dağılım — hangi saatlerde daha çok yorum yapılıyor ────────────
hourly_counts = df.groupby("hour").size().reset_index(name="yorum_sayisi")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(hourly_counts["hour"], hourly_counts["yorum_sayisi"],
              color=plt.cm.coolwarm(np.linspace(0.2, 0.8, 24)),
              edgecolor="white", linewidth=0.5)

# En yoğun saati vurgula
peak_hour = hourly_counts.loc[hourly_counts["yorum_sayisi"].idxmax(), "hour"]
bars[peak_hour].set_edgecolor("black")
bars[peak_hour].set_linewidth(2)

ax.set_xlabel("Saat (UTC)")
ax.set_ylabel("Yorum Sayısı")
ax.set_title("Saatlik Yorum Dağılımı", fontsize=14)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

print(f"📌 En yoğun yorum saati: {peak_hour}:00 UTC ({hourly_counts.loc[hourly_counts['hour']==peak_hour, 'yorum_sayisi'].values[0]:,} yorum)")

In [ ]:
# ── 5c. Haftalık trend ────────────────────────────────────────────────────────
weekly_counts = df.groupby("year_week").size().reset_index(name="yorum_sayisi")
weekly_counts["week_start"] = weekly_counts["year_week"].apply(lambda x: x.start_time)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(weekly_counts["week_start"], weekly_counts["yorum_sayisi"],
                alpha=0.3, color="#9b59b6")
ax.plot(weekly_counts["week_start"], weekly_counts["yorum_sayisi"],
        color="#9b59b6", linewidth=2)

ax.set_xlabel("Hafta")
ax.set_ylabel("Yorum Sayısı")
ax.set_title("Haftalık Yorum Trendi", fontsize=14)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Yorum:** Zaman serisi grafikleri, yorum aktivitesindeki mevsimsellik ve olası anomali noktalarını gösterir. Büyük oyun lansmanları veya indirim dönemleri, yorum sayısında ani artışlara neden olabilir.

---
## 6. Metin Analizi
Review metinlerinin uzunluk dağılımı ve en sık kullanılan kelimeler.

In [ ]:
# ── 6a. review_text uzunluğu dağılımı (histogram) ────────────────────────────
df["text_length"] = df["review_text"].fillna("").apply(len)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df["text_length"], bins=100, color="#3498db", edgecolor="white",
        linewidth=0.3, alpha=0.8)

# Ortalama ve medyan çizgileri
mean_len = df["text_length"].mean()
median_len = df["text_length"].median()
ax.axvline(mean_len, color="#e74c3c", linestyle="--", linewidth=2, label=f"Ortalama: {mean_len:.0f}")
ax.axvline(median_len, color="#f39c12", linestyle="--", linewidth=2, label=f"Medyan: {median_len:.0f}")

ax.set_xlabel("Metin Uzunluğu (karakter)")
ax.set_ylabel("Frekans")
ax.set_title("Review Metin Uzunluğu Dağılımı", fontsize=14)
ax.legend(fontsize=11)
# X ekseninde aşırı uzun metinleri kırp (percentile 99)
p99 = df["text_length"].quantile(0.99)
ax.set_xlim(0, p99)
plt.tight_layout()
plt.show()

print(f"Ortalama metin uzunluğu: {mean_len:.0f} karakter | Medyan: {median_len:.0f} karakter | Max: {df['text_length'].max():,} karakter")

In [ ]:
# ── 6b. Pozitif vs Negatif yorumlarda ortalama metin uzunluğu ─────────────────
len_by_class = df.groupby("review_score")["text_length"].agg(["mean", "median", "std"]).round(1)
len_by_class.index = ["Negatif (0)", "Pozitif (1)"]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(2)
width = 0.3

bars_mean = ax.bar(x - width/2, len_by_class["mean"], width,
                   label="Ortalama", color=["#e74c3c", "#2ecc71"], edgecolor="white")
bars_med = ax.bar(x + width/2, len_by_class["median"], width,
                  label="Medyan", color=["#c0392b", "#27ae60"], edgecolor="white", alpha=0.7)

# Bar üstüne değer yaz
for bar in bars_mean:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
for bar in bars_med:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(["Negatif (0)", "Pozitif (1)"], fontsize=12)
ax.set_ylabel("Metin Uzunluğu (karakter)")
ax.set_title("Pozitif vs Negatif Yorumlarda Metin Uzunluğu Karşılaştırması", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(len_by_class.to_string())

**Yorum:** Genellikle negatif yorumlar daha uzun olma eğilimindedir — kullanıcılar şikayetlerini daha detaylı ifade eder. Bu uzunluk farkı, modelde ek bir feature olarak kullanılabilir.

In [ ]:
# ── 6c. En sık geçen kelimeler (bar chart) ───────────────────────────────────
# Basit tokenization: küçük harf, sadece alfabetik kelimeler, 2 karakterden uzun
# İngilizce stop words listesi
STOP_WORDS = {
    "the", "a", "an", "is", "it", "and", "or", "but", "in", "on", "at", "to",
    "for", "of", "with", "as", "by", "was", "be", "are", "been", "this", "that",
    "have", "has", "had", "not", "you", "its", "from", "they", "will", "would",
    "can", "could", "do", "does", "did", "just", "more", "very", "so", "if",
    "than", "too", "also", "about", "up", "out", "all", "there", "when", "what",
    "which", "who", "how", "my", "me", "your", "we", "our", "their", "them",
    "he", "she", "his", "her", "no", "like", "get", "got", "one", "some",
    "really", "even", "much", "well", "only", "dont", "ive", "its", "im",
    "thats", "youre", "dont", "didnt", "cant", "wont", "game", "play"
}

def tokenize(text):
    """Basit tokenizer: küçük harf, alfanümerik, stop words filtre."""
    if not isinstance(text, str):
        return []
    words = re.findall(r"[a-z]+", text.lower())
    return [w for w in words if len(w) > 2 and w not in STOP_WORDS]

# Pozitif ve Negatif yorumlar için ayrı ayrı say
pos_texts = df.loc[df["review_score"] == 1, "review_text"]
neg_texts = df.loc[df["review_score"] == 0, "review_text"]

pos_words = Counter()
neg_words = Counter()
for text in pos_texts:
    pos_words.update(tokenize(text))
for text in neg_texts:
    neg_words.update(tokenize(text))

# Top-20 kelime — yan yana göster
fig, axes = plt.subplots(1, 2, figsize=(12, 7))

# Pozitif
top_pos = pos_words.most_common(20)
words_p, counts_p = zip(*top_pos)
axes[0].barh(words_p[::-1], counts_p[::-1], color="#2ecc71", edgecolor="white")
axes[0].set_title("Pozitif Yorumlarda En Sık 20 Kelime", fontsize=13)
axes[0].set_xlabel("Frekans")

# Negatif
top_neg = neg_words.most_common(20)
words_n, counts_n = zip(*top_neg)
axes[1].barh(words_n[::-1], counts_n[::-1], color="#e74c3c", edgecolor="white")
axes[1].set_title("Negatif Yorumlarda En Sık 20 Kelime", fontsize=13)
axes[1].set_xlabel("Frekans")

plt.tight_layout()
plt.show()

In [ ]:
# ── 6d. review_votes dağılımı (log scale histogram) ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Sıfır olmayan oyları filtrele (log scale için)
votes_nonzero = df.loc[df["review_votes"] > 0, "review_votes"]

ax.hist(votes_nonzero, bins=100, color="#8e44ad", edgecolor="white",
        linewidth=0.3, alpha=0.8)
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("Review Votes (log ölçek)")
ax.set_ylabel("Frekans (log ölçek)")
ax.set_title("Review Votes Dağılımı (Log-Log Scale)", fontsize=14)

# İstatistikler
mean_v = df["review_votes"].mean()
median_v = df["review_votes"].median()
max_v = df["review_votes"].max()
zero_pct = (df["review_votes"] == 0).sum() / len(df) * 100

ax.axvline(mean_v, color="#e74c3c", linestyle="--", linewidth=2,
           label=f"Ortalama: {mean_v:.1f}")
if median_v > 0:
    ax.axvline(median_v, color="#f39c12", linestyle="--", linewidth=2,
               label=f"Medyan: {median_v:.1f}")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"review_votes: Ortalama={mean_v:.1f}, Medyan={median_v:.0f}, Max={max_v:,}")
print(f"Sıfır oy alan yorumların oranı: %{zero_pct:.1f}")

**Yorum:** `review_votes` dağılımı tipik bir power-law (güç yasası) dağılımı gösterir. Yorumların büyük çoğunluğu çok az oy alırken, az sayıda yorum çok yüksek oy almıştır. Log scale bu dağılımı daha net görselleştirmemizi sağlar.

---
## 7. Korelasyon Analizi
Sayısal değişkenler arasındaki ilişkileri inceliyoruz.

In [ ]:
# ── 7a. Korelasyon matrisi ────────────────────────────────────────────────────
# text_length'i de dahil et (türetilmiş feature)
corr_cols = ["app_id", "review_score", "review_votes", "text_length"]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)

# Hücre değerlerini yaz
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.values[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=11, fontweight="bold", color=color)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(corr_cols, fontsize=11)
ax.set_title("Sayısal Değişkenler Korelasyon Matrisi (Pearson)", fontsize=14)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Korelasyon Katsayısı")
plt.tight_layout()
plt.show()

In [ ]:
# ── 7b. review_votes ile review_score ilişkisi ───────────────────────────────
# Sınıf bazında review_votes dağılımını karşılaştır
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot
vote_data = [
    df.loc[df["review_score"] == 0, "review_votes"].values,
    df.loc[df["review_score"] == 1, "review_votes"].values
]
bp = axes[0].boxplot(vote_data, labels=["Negatif (0)", "Pozitif (1)"],
                     patch_artist=True, showfliers=False)
bp["boxes"][0].set_facecolor("#e74c3c")
bp["boxes"][1].set_facecolor("#2ecc71")
axes[0].set_ylabel("Review Votes")
axes[0].set_title("Review Votes Dağılımı (Sınıf Bazında)", fontsize=13)

# Ortalama oy karşılaştırması
avg_votes = df.groupby("review_score")["review_votes"].mean()
bars = axes[1].bar(
    ["Negatif (0)", "Pozitif (1)"],
    avg_votes.values,
    color=["#e74c3c", "#2ecc71"],
    edgecolor="white", linewidth=1.5
)
for bar, val in zip(bars, avg_votes.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Ortalama Review Votes")
axes[1].set_title("Sınıf Bazında Ortalama Oy Sayısı", fontsize=13)

plt.tight_layout()
plt.show()

# Point-biserial korelasyon
from scipy.stats import pointbiserialr
corr_val, p_val = pointbiserialr(df["review_score"].dropna(), df["review_votes"].dropna())
print(f"review_score ↔ review_votes Point-Biserial Korelasyon: r={corr_val:.4f}, p={p_val:.2e}")
if p_val < 0.05:
    print("→ İstatistiksel olarak anlamlı bir ilişki tespit edildi (p < 0.05).")
else:
    print("→ İstatistiksel olarak anlamlı bir ilişki tespit edilemedi (p ≥ 0.05).")

**Yorum:** Korelasyon matrisi, değişkenler arasındaki doğrusal ilişkileri ortaya koyar. `review_votes` ile `review_score` arasındaki ilişki, kullanıcıların pozitif/negatif yorumlara farklı oranda oy verip vermediğini gösterir. Point-biserial korelasyon, ikili (binary) bir değişken ile sürekli bir değişken arasındaki ilişkiyi ölçmek için uygundur.

---
## Özet Bulgular

| Metrik | Değer |
|--------|-------|
| Toplam kayıt | `df.shape[0]` |
| Benzersiz oyun | `df["app_name"].nunique()` |
| Pozitif / Negatif oran | Pie chart'a bakınız |
| Eksik değer durumu | Heatmap'e bakınız |
| Metin uzunluğu ortalaması | Histogram'a bakınız |

**Bir sonraki adım:** Feature engineering ve model eğitimi için veriyi hazırlama.

In [ ]:
# ── Spark oturumunu kapat ─────────────────────────────────────────────────────
spark.stop()
print("Spark oturumu kapatıldı. EDA tamamlandı ✓")

# TODO: Delete this cell (duplicate)